<a href="https://colab.research.google.com/github/rajendran-official/AI_ML_COURSE_ICT/blob/main/Manappuram_NLP_Casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#path
import kagglehub

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
print("path to dataset files :",path)

100%|██████████| 242M/242M [00:01<00:00, 138MB/s]

Extracting files...


path to dataset files : /root/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2


In [2]:
import pandas as pd
import os

files = os.listdir(path)
print(files)

['Reviews.csv', 'database.sqlite', 'hashes.txt']


In [3]:
df=pd.read_csv(os.path.join(path,"Reviews.csv"))

In [4]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [5]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [6]:
df=df[['Text','Score']]
df.head()

,Text,Score
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1
2,This is a confection that has been around a fe...,4
3,If you are looking for the secret ingredient i...,2
4,Great taffy at a great price. There was a wid...,5


In [7]:
df['sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

You're seeing a SettingWithCopyWarning because you're trying to modify a slice of a DataFrame. To fix this, I'll use .loc to explicitly assign the 'sentiment' column, which ensures the operation is performed safely.


In [8]:
df.loc[:, 'sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [9]:
import re

def clean_text(text):
  text = text.lower()
  text = re.sub(r"[^a-z\s]","",text)
  return text
df.loc[:, 'review'] = df['Text'].apply(clean_text)

In [10]:
df['tokens']=df['review'].apply(lambda x: x.split())
df.head()

,Text,Score,sentiment,review,tokens
0,I have bought several of the Vitality canned d...,5,1,i have bought several of the vitality canned d...,"[i, have, bought, several, of, the, vitality, ..."
1,Product arrived labeled as Jumbo Salted Peanut...,1,0,product arrived labeled as jumbo salted peanut...,"[product, arrived, labeled, as, jumbo, salted,..."
2,This is a confection that has been around a fe...,4,1,this is a confection that has been around a fe...,"[this, is, a, confection, that, has, been, aro..."
3,If you are looking for the secret ingredient i...,2,0,if you are looking for the secret ingredient i...,"[if, you, are, looking, for, the, secret, ingr..."
4,Great taffy at a great price. There was a wid...,5,1,great taffy at a great price there was a wide...,"[great, taffy, at, a, great, price, there, was..."


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)

X=vectorizer.fit_transform(df['review'])
y=df['sentiment']

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [13]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.81      0.68      0.74     24666
           1       0.92      0.96      0.93     89025

    accuracy                           0.90    113691
   macro avg       0.86      0.82      0.84    113691
weighted avg       0.89      0.90      0.89    113691



In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence  import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(df['review'])
sequences = tokenizer.texts_to_sequences(df['review'])

X =pad_sequences(sequences,maxlen=100)
y=df['sentiment']

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense
model = Sequential()
model.add(Embedding(10000,128, input_length=100))
model.add(LSTM(64))
model.add(Dense(1,activation='sigmoid'))
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(X_train,y_train,
          epochs=3,
          batch_size=64,
          validation_split=0.1)

Epoch 1/3
1922/6396 ━━━━━━━━━━━━━━━━━━━━ 12:25 167ms/step - accuracy: 0.7811 - loss: 0.5290

In [ ]:
test = ["This product is amazing"]

seq= tokenizer.texts_to_sequences(test)
padded = pad_sequences(seq,maxlen=100)